<a href="https://colab.research.google.com/github/attabeezy/crop-guard/blob/main/notebooks/02_prepare_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropGuard — dataset audit and reproducible splits

This notebook audits all 22 CCMT classes, normalizes known folder-name errors, detects corrupt and duplicate images, creates leakage-resistant train/validation/test manifests, and packages the results for download.

**Data source:** Kaggle mirror of [Mendeley Data DOI 10.17632/bwh3zbpkpv.1](https://data.mendeley.com/datasets/bwh3zbpkpv/1), CC BY 4.0. The mirror's observed counts differ from the published paper; this notebook records that discrepancy rather than hiding it.

## 1. Install dependencies and download CCMT

A normal Colab CPU runtime is sufficient. Auditing every high-resolution image can take several minutes.

In [1]:
%pip install -q kagglehub pandas scikit-learn pillow tqdm

In [2]:
from pathlib import Path
import kagglehub

DATASET_HANDLE = "nirmalsankalana/crop-pest-and-disease-detection"
DATASET_VERSION = "1"
SEED = 2026
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
print(f"Dataset root: {dataset_root}")

100%|██████████| 1.25G/1.25G [00:14<00:00, 94.6MB/s]

Extracting files...


Dataset root: /root/.cache/kagglehub/datasets/nirmalsankalana/crop-pest-and-disease-detection/versions/1


## 2. Define the canonical class mapping

The left side must exactly match the downloaded folder names. The right side is the stable label used by training and Android.

In [3]:
FOLDER_TO_LABEL = {
    "Cashew anthracnose": ("cashew", "anthracnose"),
    "Cashew gumosis": ("cashew", "gummosis"),
    "Cashew healthy": ("cashew", "healthy"),
    "Cashew leaf miner": ("cashew", "leaf_miner"),
    "Cashew red rust": ("cashew", "red_rust"),
    "Cassava bacterial blight": ("cassava", "bacterial_blight"),
    "Cassava brown spot": ("cassava", "brown_spot"),
    "Cassava green mite": ("cassava", "green_mite"),
    "Cassava healthy": ("cassava", "healthy"),
    "Cassava mosaic": ("cassava", "mosaic"),
    "Maize fall armyworm": ("maize", "fall_armyworm"),
    "Maize grasshoper": ("maize", "grasshopper"),
    "Maize healthy": ("maize", "healthy"),
    "Maize leaf beetle": ("maize", "leaf_beetle"),
    "Maize leaf blight": ("maize", "leaf_blight"),
    "Maize leaf spot": ("maize", "leaf_spot"),
    "Maize streak virus": ("maize", "streak_virus"),
    "Tomato healthy": ("tomato", "healthy"),
    "Tomato leaf blight": ("tomato", "leaf_blight"),
    "Tomato leaf curl": ("tomato", "leaf_curl"),
    "Tomato septoria leaf spot": ("tomato", "septoria_leaf_spot"),
    "Tomato verticulium wilt": ("tomato", "verticillium_wilt"),
}

PUBLISHED_COUNTS = {
    "cashew": 6549,
    "cassava": 7508,
    "maize": 5389,
    "tomato": 5435,
}

assert len(FOLDER_TO_LABEL) == 22
assert len(set(FOLDER_TO_LABEL.values())) == 22
print("Canonical classes:", len(FOLDER_TO_LABEL))

Canonical classes: 22


## 3. Inventory and validate every image

For each file we compute SHA-256, dimensions, and a 64-bit difference hash. SHA-256 finds byte-identical files. Difference hashes are later used to identify visually near-identical images.

In [4]:
import hashlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
top_level_folders = {p.name for p in dataset_root.iterdir() if p.is_dir()}
unknown_folders = sorted(top_level_folders - set(FOLDER_TO_LABEL))
missing_folders = sorted(set(FOLDER_TO_LABEL) - top_level_folders)
if unknown_folders or missing_folders:
    raise RuntimeError(
        f"Archive structure changed. Unknown={unknown_folders}; missing={missing_folders}"
    )

files = sorted(
    path for folder in FOLDER_TO_LABEL
    for path in (dataset_root / folder).rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
print(f"Images queued for audit: {len(files):,}")

def difference_hash(image, size=8):
    gray = ImageOps.exif_transpose(image).convert("L").resize(
        (size + 1, size), Image.Resampling.LANCZOS
    )
    pixels = np.asarray(gray)
    bits = pixels[:, 1:] > pixels[:, :-1]
    value = 0
    for bit in bits.ravel():
        value = (value << 1) | int(bit)
    return f"{value:016x}"

def audit_image(path):
    relative = path.relative_to(dataset_root).as_posix()
    folder = path.relative_to(dataset_root).parts[0]
    crop, disease_class = FOLDER_TO_LABEL[folder]
    try:
        raw = path.read_bytes()
        sha256 = hashlib.sha256(raw).hexdigest()
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            width, height = ImageOps.exif_transpose(image).size
            dhash = difference_hash(image)
        return {
            "path": relative,
            "crop": crop,
            "class": disease_class,
            "label": f"{crop}|{disease_class}",
            "sha256": sha256,
            "dhash": dhash,
            "width": width,
            "height": height,
            "bytes": len(raw),
            "error": "",
        }
    except Exception as exc:
        return {
            "path": relative,
            "crop": crop,
            "class": disease_class,
            "label": f"{crop}|{disease_class}",
            "sha256": "",
            "dhash": "",
            "width": None,
            "height": None,
            "bytes": path.stat().st_size,
            "error": f"{type(exc).__name__}: {exc}",
        }

with ThreadPoolExecutor(max_workers=8) as pool:
    records = list(tqdm(pool.map(audit_image, files), total=len(files)))

inventory = pd.DataFrame(records)
corrupt = inventory[inventory["error"] != ""].copy()
valid = inventory[inventory["error"] == ""].copy().reset_index(drop=True)
print(f"Valid: {len(valid):,}; corrupt/unreadable: {len(corrupt):,}")

Images queued for audit: 25,220


  0%|          | 0/25220 [00:00<?, ?it/s]

Valid: 25,126; corrupt/unreadable: 94


## 4. Review class counts and source discrepancy

In [5]:
class_distribution = (
    valid.groupby(["crop", "class", "label"], as_index=False)
    .size()
    .rename(columns={"size": "image_count"})
    .sort_values(["crop", "class"])
)
display(class_distribution)

crop_counts = valid.groupby("crop").size().rename("observed_count").to_frame()
crop_counts["published_count"] = pd.Series(PUBLISHED_COUNTS)
crop_counts["difference"] = crop_counts["observed_count"] - crop_counts["published_count"]
display(crop_counts)

if (crop_counts["difference"] != 0).any():
    print("WARNING: mirror counts differ from the published Mendeley totals. "
          "Keep this discrepancy in the data documentation.")

,crop,class,label,image_count
0,cashew,anthracnose,cashew|anthracnose,1729
1,cashew,gummosis,cashew|gummosis,392
2,cashew,healthy,cashew|healthy,1368
3,cashew,leaf_miner,cashew|leaf_miner,1378
4,cashew,red_rust,cashew|red_rust,1682
5,cassava,bacterial_blight,cassava|bacterial_blight,2614
6,cassava,brown_spot,cassava|brown_spot,1481
7,cassava,green_mite,cassava|green_mite,1015
8,cassava,healthy,cassava|healthy,1193
9,cassava,mosaic,cassava|mosaic,1205


,observed_count,published_count,difference
crop,,,
cashew,6549,6549,0
cassava,7508,7508,0
maize,5289,5389,-100
tomato,5780,5435,345


## 5. Detect exact and likely near-duplicates

Near-duplicate candidates use a difference-hash Hamming distance of at most 4. Eight independent 8-bit buckets avoid comparing every possible image pair while guaranteeing that hashes within the threshold share a bucket. Candidate comparisons are restricted to the same class; exact duplicates across different labels are reported as conflicts.

In [6]:
from collections import defaultdict
from itertools import combinations

exact_label_counts = valid.groupby("sha256")["label"].nunique()
conflicting_hashes = set(exact_label_counts[exact_label_counts > 1].index)
label_conflicts = valid[valid["sha256"].isin(conflicting_hashes)].copy()
if len(label_conflicts):
    print(f"CRITICAL: {len(conflicting_hashes)} exact hashes occur under multiple labels.")

parent = list(range(len(valid)))

def find(item):
    while parent[item] != item:
        parent[item] = parent[parent[item]]
        item = parent[item]
    return item

def union(left, right):
    root_left, root_right = find(left), find(right)
    if root_left != root_right:
        parent[root_right] = root_left

# Exact duplicate groups always stay together.
for indices in valid.groupby("sha256").indices.values():
    first = indices[0]
    for other in indices[1:]:
        union(first, other)

hash_values = [int(value, 16) for value in valid["dhash"]]
near_pairs = set()
for label, label_indices in valid.groupby("label").indices.items():
    buckets = defaultdict(list)
    for index in label_indices:
        value = hash_values[index]
        for chunk in range(8):
            buckets[(chunk, (value >> (chunk * 8)) & 0xFF)].append(index)
    candidates = set()
    for members in buckets.values():
        if len(members) > 1:
            candidates.update(combinations(members, 2))
    for left, right in candidates:
        distance = (hash_values[left] ^ hash_values[right]).bit_count()
        if distance <= 4:
            pair = (min(left, right), max(left, right), distance)
            near_pairs.add(pair)
            union(left, right)

valid["duplicate_group"] = [f"g{find(i):06d}" for i in range(len(valid))]
near_duplicate_pairs = pd.DataFrame(
    [
        {
            "left_path": valid.at[left, "path"],
            "right_path": valid.at[right, "path"],
            "label": valid.at[left, "label"],
            "hamming_distance": distance,
        }
        for left, right, distance in sorted(near_pairs)
    ]
)
duplicate_sizes = valid.groupby("duplicate_group").size()
exact_duplicates = valid[valid.duplicated("sha256", keep=False)].copy()
eligible = (
    valid[~valid["sha256"].isin(conflicting_hashes)]
    .drop_duplicates("sha256", keep="first")
    .copy()
    .reset_index(drop=True)
)
print(f"Exact duplicate files beyond first copy: {valid.duplicated('sha256').sum():,}")
print(f"Unique images eligible for splitting: {len(eligible):,}")
print(f"Likely near-duplicate pairs: {len(near_duplicate_pairs):,}")
print(f"Multi-image duplicate groups: {(duplicate_sizes > 1).sum():,}")

CRITICAL: 224 exact hashes occur under multiple labels.
Exact duplicate files beyond first copy: 831
Unique images eligible for splitting: 24,071
Likely near-duplicate pairs: 1,225
Multi-image duplicate groups: 1,098


## 6. Create grouped 70/10/20 splits

All members of an exact/near-duplicate group remain in one split. Splitting is performed independently per class to retain representation of every class.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

if len(corrupt):
    print("Corrupt images are excluded and listed in corrupt_images.csv.")
if len(label_conflicts):
    print("Cross-label duplicate conflicts are excluded from every split and retained in the report.")

split_parts = {"train": [], "validation": [], "test": []}
for label, subset in eligible.groupby("label", sort=True):
    subset = subset.reset_index(drop=True)
    groups = subset["duplicate_group"]
    if groups.nunique() < 10:
        raise RuntimeError(f"Too few independent groups for {label}: {groups.nunique()}")
    outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_val_idx, test_idx = next(outer.split(subset, groups=groups))
    train_val = subset.iloc[train_val_idx].reset_index(drop=True)
    inner = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED + 1)
    train_idx, validation_idx = next(
        inner.split(train_val, groups=train_val["duplicate_group"])
    )
    split_parts["train"].append(train_val.iloc[train_idx])
    split_parts["validation"].append(train_val.iloc[validation_idx])
    split_parts["test"].append(subset.iloc[test_idx])

splits = {
    name: pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
    for name, parts in split_parts.items()
}
for name, frame in splits.items():
    frame["split"] = name
    print(f"{name:10}: {len(frame):6,} images, {frame['duplicate_group'].nunique():6,} groups")

Corrupt images are excluded and listed in corrupt_images.csv.
Cross-label duplicate conflicts are excluded from every split and retained in the report.
train     : 16,845 images, 16,541 groups
validation:  2,415 images,  2,374 groups
test      :  4,811 images,  4,739 groups


## 7. Verify split invariants

In [8]:
all_splits = pd.concat(splits.values(), ignore_index=True)
assert len(all_splits) == len(eligible)
assert all_splits["path"].nunique() == len(eligible)
assert all_splits.groupby("sha256")["split"].nunique().max() == 1
assert all_splits.groupby("duplicate_group")["split"].nunique().max() == 1
assert all_splits.groupby("label")["split"].nunique().min() == 3

split_distribution = pd.crosstab(all_splits["label"], all_splits["split"])
split_distribution["total"] = split_distribution.sum(axis=1)
display(split_distribution)
split_ratios = split_distribution[["train", "validation", "test"]].div(
    split_distribution["total"], axis=0
)
target_ratios = pd.Series({"train": 0.70, "validation": 0.10, "test": 0.20})
max_ratio_deviation = float((split_ratios - target_ratios).abs().to_numpy().max())
display(split_ratios.round(3))
if max_ratio_deviation > 0.05:
    print(f"REVIEW: largest class/split ratio deviation is {max_ratio_deviation:.1%}.")
print("PASS: every eligible unique image appears once; no duplicate group crosses splits; "
      "all 22 classes occur in all three splits.")

split,test,train,validation,total
label,,,,
cashew|anthracnose,345,1210,174,1729
cashew|gummosis,79,274,39,392
cashew|healthy,268,961,139,1368
cashew|leaf_miner,275,965,138,1378
cashew|red_rust,336,1175,170,1681
cassava|bacterial_blight,518,1814,260,2592
cassava|brown_spot,292,1027,146,1465
cassava|green_mite,205,709,100,1014
cassava|healthy,240,834,119,1193


split,train,validation,test
label,,,
cashew|anthracnose,0.700,0.101,0.200
cashew|gummosis,0.699,0.099,0.202
cashew|healthy,0.702,0.102,0.196
cashew|leaf_miner,0.700,0.100,0.200
cashew|red_rust,0.699,0.101,0.200
cassava|bacterial_blight,0.700,0.100,0.200
cassava|brown_spot,0.701,0.100,0.199
cassava|green_mite,0.699,0.099,0.202
cassava|healthy,0.699,0.100,0.201


PASS: every eligible unique image appears once; no duplicate group crosses splits; all 22 classes occur in all three splits.


## 8. Export and download the audit bundle

The ZIP contains metadata and manifests only—no copyrighted image files: `train.csv`, `validation.csv`, `test.csv`, `data_audit.json`, `image_inventory.csv`, `class_distribution.csv`, `split_distribution.csv`, `corrupt_images.csv`, `label_conflicts.csv`, `exact_duplicates.csv`, `near_duplicate_pairs.csv`, and `split_hashes.json`.

In [9]:
import json
import platform
import shutil
from datetime import datetime, timezone

output_dir = Path("/content/cropguard_data_prep")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True)

manifest_columns = [
    "path", "crop", "class", "label", "sha256", "dhash",
    "duplicate_group", "width", "height", "bytes", "split",
]
for name, frame in splits.items():
    frame[manifest_columns].to_csv(output_dir / f"{name}.csv", index=False)

inventory.to_csv(output_dir / "image_inventory.csv", index=False)
class_distribution.to_csv(output_dir / "class_distribution.csv", index=False)
split_distribution.to_csv(output_dir / "split_distribution.csv")
corrupt.to_csv(output_dir / "corrupt_images.csv", index=False)
label_conflicts.to_csv(output_dir / "label_conflicts.csv", index=False)
exact_duplicates.to_csv(output_dir / "exact_duplicates.csv", index=False)
near_duplicate_pairs.to_csv(output_dir / "near_duplicate_pairs.csv", index=False)

file_hashes = {}
for csv_path in sorted(output_dir.glob("*.csv")):
    file_hashes[csv_path.name] = hashlib.sha256(csv_path.read_bytes()).hexdigest()
(output_dir / "split_hashes.json").write_text(json.dumps(file_hashes, indent=2))

audit = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "kaggle_handle": DATASET_HANDLE,
        "kaggle_version": DATASET_VERSION,
        "authoritative_doi": "10.17632/bwh3zbpkpv.1",
        "license": "CC BY 4.0",
    },
    "seed": SEED,
    "class_count": int(valid["label"].nunique()),
    "discovered_images": len(files),
    "valid_images": len(valid),
    "unique_images_in_splits": len(eligible),
    "corrupt_images": len(corrupt),
    "exact_duplicate_copies": int(valid.duplicated("sha256").sum()),
    "near_duplicate_pairs": len(near_duplicate_pairs),
    "cross_label_exact_conflicts": len(conflicting_hashes),
    "published_counts": PUBLISHED_COUNTS,
    "observed_counts": {k: int(v) for k, v in valid.groupby("crop").size().items()},
    "split_counts": {k: len(v) for k, v in splits.items()},
    "max_class_split_ratio_deviation": max_ratio_deviation,
    "python": platform.python_version(),
    "status": "pass" if (
        not len(corrupt) and not len(label_conflicts) and max_ratio_deviation <= 0.05
    ) else "review_required",
}
(output_dir / "data_audit.json").write_text(json.dumps(audit, indent=2))

archive = shutil.make_archive("/content/cropguard-data-prep", "zip", output_dir)
print(f"Created {archive} ({Path(archive).stat().st_size / 1_000_000:.1f} MB)")
display(audit)

Created /content/cropguard-data-prep.zip (3.1 MB)


{'created_utc': '2026-06-27T02:34:16.381049+00:00',
 'source': {'kaggle_handle': 'nirmalsankalana/crop-pest-and-disease-detection',
  'kaggle_version': '1',
  'authoritative_doi': '10.17632/bwh3zbpkpv.1',
  'license': 'CC BY 4.0'},
 'seed': 2026,
 'class_count': 22,
 'discovered_images': 25220,
 'valid_images': 25126,
 'unique_images_in_splits': 24071,
 'corrupt_images': 94,
 'exact_duplicate_copies': 831,
 'near_duplicate_pairs': 1225,
 'cross_label_exact_conflicts': 224,
 'published_counts': {'cashew': 6549,
  'cassava': 7508,
  'maize': 5389,
  'tomato': 5435},
 'observed_counts': {'cashew': 6549,
  'cassava': 7508,
  'maize': 5289,
  'tomato': 5780},
 'split_counts': {'train': 16845, 'validation': 2415, 'test': 4811},
 'max_class_split_ratio_deviation': 0.018199608610567575,
 'python': '3.12.13',
 'status': 'review_required'}

In [10]:
from google.colab import files
files.download("/content/cropguard-data-prep.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What to do next

Download `cropguard-data-prep.zip` and keep it unchanged. Do not upload image files to GitHub. Before training, review `data_audit.json`, `label_conflicts.csv`, `corrupt_images.csv`, the class distribution, and a sample of `near_duplicate_pairs.csv`.